# CRYSTAL — Phase 4: domain-adaptive pretraining (DAPT) on a Kaggle GPU

Settings: **Accelerator = GPU T4**, **Internet = On**, and **Add-ons -> Secrets -> `HF_TOKEN`** = a
Hugging Face *read* token of the account that accepted the CryCeleb2023 terms (attach it to this
notebook). Keep this notebook **private** (CryCeleb is CC BY-NC-ND; the checkpoint must not be shared).
Then **Save Version -> Save & Run All**. Download `crystal_results.zip` (logs/metrics, no audio) from the
Output tab; `dapt_checkpoint.zip` (the pretrained model, ~0.4 GB) is kept there too for later phases.

In [ ]:
DAPT_CONFIG = "configs/dapt/D01_w2v2_cryceleb.yaml"
DAPT_MAX_STEPS = None                                   # None = config value (10k); e.g. 500 for a smoke test
CONFIGS = [                                             # run after DAPT with init_checkpoint dapt:D01
    "configs/exp/E20_dapt_w2v2_probe.yaml",
    "configs/exp/E21_dapt_w2v2_ft.yaml",
]
EXTRA_ARGS = ""
SAVE_CHECKPOINT = True
REPO = "https://github.com/IdrisKulubi/crystal.git"
BRANCH = "main"

In [ ]:
import os, subprocess, sys, json, hashlib, pathlib
# work in /tmp (NOT /kaggle/working): Kaggle saves /kaggle/working as notebook output, and we
# never want the audio in data/ to end up there. Only crystal_results.zip is written to output.
os.chdir("/tmp")
if not pathlib.Path("crystal").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, "crystal"], check=True)
os.chdir("/tmp/crystal")
subprocess.run(["git", "log", "--oneline", "-1"], check=True)
# Kaggle ships torchao 0.10, which recent peft refuses to import alongside (breaks LoRA); we don't use it
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "librosa", "soundfile", "pyyaml", "scikit-learn", "pandas", "transformers>=4.44", "peft>=0.12"], check=True)
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available() or "--debug" in EXTRA_ARGS, (
    "No GPU. Stop the session, set Settings -> Accelerator -> GPU T4 x2 / P100 (needs a phone-verified "
    "account), then run again. Training on CPU here would take days.")

In [ ]:
# data: same public sources and the same pipeline as locally
os.makedirs("data/raw", exist_ok=True); os.makedirs("baselines", exist_ok=True)
if not pathlib.Path("data/raw/donateacry-corpus").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/gveres/donateacry-corpus", "data/raw/donateacry-corpus"], check=True)
if not pathlib.Path("baselines/DeepInfant").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/skytells-research/DeepInfant", "baselines/DeepInfant"], check=True)
if not pathlib.Path("data/raw/ESC-50").exists():   # noise augmentation pool (public, CC BY-NC), training only
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/karolpiczak/ESC-50", "data/raw/ESC-50"], check=True)
expected = json.load(open("results/splits_report.json"))["folds_file_sha256"]
exp_r = json.load(open("results/splits_report.json"))["randomsplit_file_sha256"]   # read BEFORE make_splits rewrites it
for step in ["prepare_data", "dedup", "make_splits"]:
    subprocess.run([sys.executable, "-m", f"src.{step}"], check=True)
got = hashlib.sha256(open("data/metadata_folds.csv", "rb").read()).hexdigest()
assert got == expected, f"folds file differs from the committed one: {got} != {expected}"
print("folds file identical to the committed protocol:", got[:16])
got_r = hashlib.sha256(open("data/metadata_randomsplit.csv", "rb").read()).hexdigest()
assert got_r == exp_r, f"random-split file differs from the committed one: {got_r} != {exp_r}"
subprocess.run(["git", "checkout", "--", "results/"], check=True)  # keep committed results untouched

In [ ]:
# CryCeleb (gated): token from Kaggle Secrets, audio stays in /tmp
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import snapshot_download
snapshot_download("Ubenwa/CryCeleb2023", repo_type="dataset", local_dir="data/raw/cryceleb",
                  allow_patterns=["audio.zip", "metadata.csv", "README.md"])
expected_h = json.load(open("results/dapt_stats.json"))["hours_unlabelled"]
subprocess.run([sys.executable, "-m", "src.build_dapt_corpus"], check=True)
got_h = json.load(open("results/dapt_stats.json"))["hours_unlabelled"]
assert got_h == expected_h, f"DAPT corpus differs from the committed one: {got_h} h != {expected_h} h"
subprocess.run(["git", "checkout", "--", "results/"], check=True)

In [ ]:
cmd = [sys.executable, "-m", "src.dapt", "--config", DAPT_CONFIG]
if DAPT_MAX_STEPS: cmd += ["--max-steps", str(DAPT_MAX_STEPS)]
print(" ".join(cmd), flush=True)
subprocess.run(cmd, check=False)
dapt_runs = sorted(p.name for p in pathlib.Path("results/dapt").glob("*") if p.is_dir())
print("DAPT runs:", dapt_runs)
for d in dapt_runs: print(d, json.load(open(f"results/dapt/{d}/status.json")))

In [ ]:
reg = pathlib.Path("results/registry.jsonl")
n_before = len(reg.read_text().splitlines()) if reg.exists() else 0
for cfg in CONFIGS:
    cmd = [sys.executable, "-m", "src.train", "--config", cfg] + EXTRA_ARGS.split()
    print(" ".join(cmd), flush=True)
    subprocess.run(cmd, check=False)   # failures are logged in the registry, keep going

In [ ]:
import zipfile
lines = reg.read_text().splitlines()[n_before:]
out = pathlib.Path("/kaggle/working/crystal_results.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr("registry_new.jsonl", "\n".join(lines) + ("\n" if lines else ""))
    for line in lines:
        rid = json.loads(line)["run_id"]
        for f in pathlib.Path("results/runs", rid).rglob("*"):
            if f.is_file():
                z.write(f, pathlib.Path("runs", rid, f.relative_to(pathlib.Path("results/runs", rid))))
    for d in dapt_runs:
        for f in pathlib.Path("results/dapt", d).rglob("*"):
            if f.is_file():
                z.write(f, pathlib.Path("dapt", d, f.relative_to(pathlib.Path("results/dapt", d))))
print(out, round(out.stat().st_size / 1e6, 1), "MB;", len(lines), "registry lines;", len(dapt_runs), "DAPT runs")
for line in lines: r = json.loads(line); print(r["run_id"], r["status"], r.get("macro_f1_mean"))
if SAVE_CHECKPOINT:
    for d in dapt_runs:
        ck = pathlib.Path("checkpoints", d)
        if ck.is_dir():
            with zipfile.ZipFile("/kaggle/working/dapt_checkpoint.zip", "w", zipfile.ZIP_STORED) as z:
                for f in ck.rglob("*"):
                    z.write(f, pathlib.Path(d, f.relative_to(ck)))
            print("saved /kaggle/working/dapt_checkpoint.zip")